In [8]:
%run clasificador_intenciones.py



In [9]:

import sys
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from clasificador_intenciones import intenciones



BASE_DIR = Path.cwd().resolve()

while not (BASE_DIR / "models" / "clasificador_intenciones.py").is_file():
    if BASE_DIR == BASE_DIR.parent:
        raise FileNotFoundError(
            "Abre el notebook desde la carpeta DinoBot o una de sus subcarpetas."
        )

    BASE_DIR = BASE_DIR.parent

sys.path.insert(0, str(BASE_DIR / "src" / "dinobot"))

RUTA_EXCEL = BASE_DIR / "data"  / "prueba.xlsx"
X, y = intenciones(RUTA_EXCEL)

modelos = {
    "SVM lineal": make_pipeline(TfidfVectorizer(), LinearSVC()),
    "Naive Bayes multinomial": make_pipeline(
        TfidfVectorizer(), MultinomialNB()
    ),
}

particiones = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metricas_evaluar=[
    "accuracy", 
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "f1_weighted",
]

resultados = []
for nombre, modelo in modelos.items():
    metricas = cross_validate(
        modelo,
        X,
        y,
        cv=particiones,
        scoring=metricas_evaluar,
        return_train_score=True,
    )

    resultados.append({
        "Modelo": nombre,
        "Accuracy": metricas["test_accuracy"].mean(),
        "F1 Macro (Test)": metricas["test_f1_macro"].mean(),
        "F1 Macro (Std)": metricas["test_f1_macro"].std(),
        "F1 Macro (Train)": metricas["train_f1_macro"].mean(),
        "Precision Macro": metricas["test_precision_macro"].mean(),
        "Recall Macro": metricas["test_recall_macro"].mean(),
        "F1 Weighted": metricas["test_f1_weighted"].mean(),
        "Inferencia (ms)": metricas["score_time"].mean() * 1000,
    })

pd.DataFrame(resultados).round(4)

,Modelo,Accuracy,F1 Macro (Test),F1 Macro (Std),F1 Macro (Train),Precision Macro,Recall Macro,F1 Weighted,Inferencia (ms)
0,SVM lineal,0.6667,0.6437,0.0239,1.0000,0.6677,0.6667,0.6437,7.1049
1,Naive Bayes multinomial,0.6300,0.6107,0.0358,0.9908,0.6700,0.6300,0.6107,7.8259


In [10]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict

etiquetas = sorted(y.unique())

for nombre, modelo in modelos.items():
    predicciones = cross_val_predict(
        modelo,
        X,
        y,
        cv=particiones,
    )

    matriz = pd.DataFrame(
        confusion_matrix(y, predicciones, labels=etiquetas),
        index=pd.Index(etiquetas, name="Intención real"),
        columns=pd.Index(etiquetas, name="Intención predicha"),
    )

    evaluacion = pd.DataFrame({
        "Pregunta": X.to_numpy(),
        "Intención real": y.to_numpy(),
        "Intención predicha": predicciones,
    })

    errores = evaluacion[
        evaluacion["Intención real"] != evaluacion["Intención predicha"]
    ]

    print(nombre)
    display(matriz)
    display(errores)

SVM lineal


Intención predicha,consultar_alimentacion,consultar_anatomia,consultar_crecimiento,consultar_descubrimiento,consultar_epoca,consultar_habitat,consultar_habitos,consultar_locomocion,consultar_tamano,consultar_ubicacion_actual
Intención real,,,,,,,,,,
consultar_alimentacion,20,0,0,0,0,4,2,0,3,1
consultar_anatomia,0,24,0,0,0,0,0,2,3,1
consultar_crecimiento,1,1,15,0,1,1,1,2,6,2
consultar_descubrimiento,1,0,0,25,2,0,0,0,0,2
consultar_epoca,0,0,2,6,16,2,0,1,1,2
consultar_habitat,3,0,0,0,2,23,1,0,0,1
consultar_habitos,4,3,5,1,0,1,8,4,2,2
consultar_locomocion,1,1,1,0,1,0,3,19,2,2
consultar_tamano,1,1,3,1,0,1,0,2,21,0


,Pregunta,Intención real,Intención predicha
4,¿Era herbívoro o carnívoro?,consultar_alimentacion,consultar_tamano
12,Explícame qué tipo de comida buscaba.,consultar_alimentacion,consultar_habitos
14,Me interesa saber si comía carne.,consultar_alimentacion,consultar_tamano
18,¿Consumía vegetación?,consultar_alimentacion,consultar_habitat
19,Necesito una estimación de su altura.,consultar_tamano,consultar_alimentacion
...,...,...,...
289,¿Se confundieron sus primeros fósiles con los ...,consultar_descubrimiento,consultar_ubicacion_actual
291,¿Qué diferencias había entre las crías y los a...,consultar_crecimiento,consultar_habitos
296,¿Llegó a vivir en el Paleógeno?,consultar_epoca,consultar_habitat
298,¿Cómo se desarrollaba desde que salía del huevo?,consultar_crecimiento,consultar_habitat


Naive Bayes multinomial


Intención predicha,consultar_alimentacion,consultar_anatomia,consultar_crecimiento,consultar_descubrimiento,consultar_epoca,consultar_habitat,consultar_habitos,consultar_locomocion,consultar_tamano,consultar_ubicacion_actual
Intención real,,,,,,,,,,
consultar_alimentacion,20,1,0,0,0,3,0,0,5,1
consultar_anatomia,1,23,0,0,0,1,0,0,2,3
consultar_crecimiento,1,1,15,1,1,2,1,2,4,2
consultar_descubrimiento,2,0,1,25,1,0,0,0,0,1
consultar_epoca,0,0,2,4,16,2,0,0,1,5
consultar_habitat,4,0,0,0,1,20,2,0,1,2
consultar_habitos,5,3,5,1,0,1,5,4,3,3
consultar_locomocion,1,0,3,1,0,0,2,17,4,2
consultar_tamano,3,2,3,2,0,0,0,1,19,0


,Pregunta,Intención real,Intención predicha
4,¿Era herbívoro o carnívoro?,consultar_alimentacion,consultar_tamano
5,Me gustaría conocer sus dimensiones.,consultar_tamano,consultar_alimentacion
6,Quiero conocer su dieta.,consultar_alimentacion,consultar_tamano
13,Quiero saber cuánto medía de la cabeza a la cola.,consultar_tamano,consultar_anatomia
14,Me interesa saber si comía carne.,consultar_alimentacion,consultar_tamano
...,...,...,...
289,¿Se confundieron sus primeros fósiles con los ...,consultar_descubrimiento,consultar_ubicacion_actual
291,¿Qué diferencias había entre las crías y los a...,consultar_crecimiento,consultar_habitat
296,¿Llegó a vivir en el Paleógeno?,consultar_epoca,consultar_habitat
298,¿Cómo se desarrollaba desde que salía del huevo?,consultar_crecimiento,consultar_habitat


In [11]:
df = pd.read_excel(RUTA_EXCEL)
df

,id,texto,intencion,especie,periodo
0,DB001,¿Qué comía?,consultar_alimentacion,NaN,NaN
1,DB002,¿Cuánto medía de largo?,consultar_tamano,NaN,NaN
2,DB003,¿De qué se alimentaba?,consultar_alimentacion,NaN,NaN
3,DB004,¿Qué tamaño alcanzaba un adulto?,consultar_tamano,NaN,NaN
4,DB005,¿Era herbívoro o carnívoro?,consultar_alimentacion,NaN,NaN
...,...,...,...,...,...
295,DB289,¿En qué publicación apareció su nombre por pri...,consultar_descubrimiento,NaN,NaN
296,DB173,¿Llegó a vivir en el Paleógeno?,consultar_epoca,NaN,NaN
297,DB061,¿Cómo era el paisaje donde vivía?,consultar_habitat,NaN,NaN
298,DB256,¿Cómo se desarrollaba desde que salía del huevo?,consultar_crecimiento,NaN,NaN


In [12]:
especies=df["especie"].value_counts();
especies

Series([], Name: count, dtype: int64)